# 01 — Data Inventory

**Purpose**: Inspect every dataset file — schema, row counts, memory usage, missing values, categorical distributions, and join integrity.

**Output**: `outputs/reports/data_inventory.md`

> **Rule**: Do NOT fabricate results. Every number in this notebook comes from the actual data.
> If a file is missing, the notebook will report it clearly rather than failing silently.

In [ ]:
import sys
import os
import logging
from pathlib import Path

import numpy as np
import pandas as pd

# ── Setup project path ──────────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ALL_RAW_FILES, REPORTS_DIR, FIGURES_DIR, TABLES_DIR, DATA_PROCESSED,
    print_file_status, ensure_dirs,
)
from src.data_loading import (
    load_players, load_games, load_career_successes,
    load_combine_results, load_combine_tracking, load_player_play,
    quick_schema,
)
from src.player_mapping import validate_player_joins, print_join_report, validate_combine_tracking
from src.visualization import plot_missing_data_heatmap

# ── Configure logging ───────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s | %(name)s | %(message)s',
)
log = logging.getLogger('01_data_inventory')

# ── Create output directories ───────────────────────────────────────────────
ensure_dirs()

print('Project root:', PROJECT_ROOT)
print('Python version:', sys.version)
print('Pandas version:', pd.__version__)
print('NumPy version:', np.__version__)
print()
print_file_status()

## 1. Quick Schema Peek

Before loading full files, inspect column names and sample values from each file.

In [ ]:
schemas = {}

for name, path in ALL_RAW_FILES.items():
    if not path.exists():
        print(f'✗ {name}: FILE NOT FOUND — {path}')
        continue
    head, dtypes = quick_schema(path, n_rows=3)
    schemas[name] = dtypes
    print(f'\n{'='*60}')
    print(f'  {name}  ({path.stat().st_size / 1_048_576:.1f} MB)')
    print(f'  Columns ({len(head.columns)}):')
    display(dtypes)
    print(f'  Head:')
    display(head)

## 2. Load Small Files Fully

In [ ]:
# Load all small files
players = None
games = None
career_successes = None
combine_results = None

try:
    players = load_players()
    print(f'players: {len(players):,} rows × {players.shape[1]} cols')
    print(players.dtypes.to_string())
except FileNotFoundError as e:
    print(f'ERROR: {e}')

try:
    games = load_games()
    print(f'\ngames: {len(games):,} rows × {games.shape[1]} cols')
    display(games.head(3))
except FileNotFoundError as e:
    print(f'ERROR: {e}')

try:
    career_successes = load_career_successes()
    print(f'\ncareer_successes: {len(career_successes):,} rows × {career_successes.shape[1]} cols')
except FileNotFoundError as e:
    print(f'ERROR: {e}')

try:
    combine_results = load_combine_results(cols=None)  # Load ALL columns to see what's there
    print(f'\ncombine_results: {len(combine_results):,} rows × {combine_results.shape[1]} cols')
    display(combine_results.head(3))
except FileNotFoundError as e:
    print(f'ERROR: {e}')

## 3. Players — Detailed Profile

In [ ]:
if players is not None:
    print('=== PLAYERS TABLE ===')
    print(f'Shape: {players.shape}')
    print(f'\nUnique nfl_id: {players["nfl_id"].nunique()}')
    print(f'Null nfl_id: {players["nfl_id"].isna().sum()}')
    print(f'nfl_id unique? {players["nfl_id"].is_unique}')
    print()

    # Missing values
    miss = players.isnull().sum()
    miss = miss[miss > 0]
    if not miss.empty:
        print('Columns with missing values:')
        print((miss / len(players) * 100).to_string())
    else:
        print('No missing values in players.')

    print()

    # Position distribution
    if 'position' in players.columns:
        print('Position value counts:')
        display(players['position'].value_counts().to_frame())

    # Numeric summary
    print('\nNumeric summary:')
    display(players.describe())

## 4. Combine Results — Detailed Profile

In [ ]:
if combine_results is not None:
    print('=== COMBINE RESULTS ===')
    print(f'Shape: {combine_results.shape}')
    print(f'Columns: {combine_results.columns.tolist()}')
    print()

    # Missing values per column (as percentage)
    miss_pct = (combine_results.isnull().sum() / len(combine_results) * 100).sort_values(ascending=False)
    miss_pct = miss_pct[miss_pct > 0]
    print('Missing values (%):')
    display(miss_pct.to_frame('missing_pct'))

    print('\nNumeric summary:')
    display(combine_results.describe())

    plot_missing_data_heatmap(combine_results, title='Missing Data — combine_results', filename='missing_combine_results.png')
    plt_shown = True

## 5. Combine Tracking — Schema + Validation

In [ ]:
combine_tracking = None

try:
    combine_tracking = load_combine_tracking()
    print(f'combine_tracking: {len(combine_tracking):,} rows × {combine_tracking.shape[1]} cols')
    print(f'Memory: {combine_tracking.memory_usage(deep=True).sum() / 1_048_576:.1f} MB')
    print()
    display(combine_tracking.head())

    # Validate structure
    val = validate_combine_tracking(combine_tracking)
    print('\nCombine Tracking Validation:')
    for k, v in val.items():
        if k != 'complete_player_ids':
            print(f'  {k}: {v}')

except FileNotFoundError as e:
    print(f'ERROR: {e}')

In [ ]:
if combine_tracking is not None:
    # Drill type distribution
    print('=== DRILL TYPE DISTRIBUTION ===')
    if 'drill_type' in combine_tracking.columns:
        drill_counts = combine_tracking.groupby('drill_type').agg(
            n_rows=('nfl_id', 'count'),
            n_players=('nfl_id', 'nunique'),
        ).sort_values('n_players', ascending=False)
        display(drill_counts)

    # Drill name distribution
    if 'drill_name' in combine_tracking.columns:
        print('\nTop drill names:')
        display(combine_tracking['drill_name'].value_counts().head(20).to_frame())

    # Attempts per player per drill
    if 'attempt' in combine_tracking.columns and 'drill_type' in combine_tracking.columns:
        attempts_per_player = (
            combine_tracking.groupby(['nfl_id', 'drill_type'])['attempt']
            .nunique()
            .reset_index(name='n_attempts')
        )
        print('\nAttempts per player per drill (distribution):')
        display(attempts_per_player['n_attempts'].describe())
        display(attempts_per_player.groupby('drill_type')['n_attempts'].describe())

    # Frames per attempt
    keys = [c for c in ['nfl_id', 'drill_type', 'attempt'] if c in combine_tracking.columns]
    frames_per_attempt = combine_tracking.groupby(keys).size().reset_index(name='n_frames')
    print('\nFrames per attempt (distribution):')
    display(frames_per_attempt['n_frames'].describe())

    plot_missing_data_heatmap(combine_tracking, title='Missing Data — combine_tracking', filename='missing_combine_tracking.png')

## 6. Player Play — Schema + Basic Stats

In [ ]:
player_play = None

try:
    player_play = load_player_play()
    print(f'player_play: {len(player_play):,} rows × {player_play.shape[1]} cols')
    print(f'Memory: {player_play.memory_usage(deep=True).sum() / 1_048_576:.1f} MB')
    print(f'Unique players: {player_play["nfl_id"].nunique()}')
    print(f'Unique games: {player_play["game_id"].nunique() if "game_id" in player_play.columns else "N/A"}')
    display(player_play.head(3))

    # Missing values
    miss_pct = (player_play.isnull().sum() / len(player_play) * 100).sort_values(ascending=False)
    miss_pct = miss_pct[miss_pct > 0]
    print('\nMissing values (%):')
    display(miss_pct.to_frame('missing_pct'))

    if 'position' in player_play.columns:
        print('\nPosition distribution in player_play:')
        display(player_play['position'].value_counts().to_frame())

except FileNotFoundError as e:
    print(f'ERROR: {e}')

## 7. Game Tracking — Schema Only (No Full Load)

In [ ]:
from src.config import GAME_TRACKING_FILES

print('=== GAME TRACKING FILES ===')
for season, path in GAME_TRACKING_FILES.items():
    if path.exists():
        size_mb = path.stat().st_size / 1_048_576
        head, dtypes = quick_schema(path, n_rows=3)
        print(f'\ngame_tracking_{season}: {size_mb:.0f} MB | {len(head.columns)} columns')
        display(dtypes)
    else:
        print(f'game_tracking_{season}: FILE NOT FOUND')

## 8. Join Validation

In [ ]:
if players is not None:
    join_report = validate_player_joins(
        players=players,
        combine_results=combine_results,
        combine_tracking=combine_tracking,
        player_play=player_play,
        career_successes=career_successes,
    )
    print_join_report(join_report)
    
    # Also report how many combine players have NFL outcomes
    if combine_tracking is not None and player_play is not None:
        ct_ids = set(combine_tracking['nfl_id'].dropna().unique())
        pp_ids = set(player_play['nfl_id'].dropna().unique())
        both = ct_ids & pp_ids
        print(f'Players with BOTH combine tracking AND player_play data: {len(both)}')
        print(f'  This is the core analysis population.')
else:
    print('Cannot run join validation — players table not loaded.')

## 9. Numeric Summary for All Datasets

In [ ]:
import matplotlib.pyplot as plt

for name, df in [
    ('players', players),
    ('combine_results', combine_results),
    ('combine_tracking', combine_tracking),
    ('player_play', player_play),
    ('career_successes', career_successes),
    ('games', games),
]:
    if df is None:
        continue
    print(f'\n{"="*60}')
    print(f'  {name} — Numeric summary')
    print(f'{"="*60}')
    num_cols = df.select_dtypes(include=np.number).columns
    if len(num_cols) > 0:
        display(df[num_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    
    cat_cols = df.select_dtypes(include='object').columns[:5]  # limit to 5
    for col in cat_cols:
        vc = df[col].value_counts()
        print(f'\n  {col} (top 10):')
        display(vc.head(10).to_frame())

## 10. Generate data_inventory.md Report

In [ ]:
def generate_inventory_report(
    players, combine_results, combine_tracking, player_play,
    career_successes, games, join_report,
) -> str:
    """Generate a markdown inventory report string."""
    from datetime import datetime
    from src.config import ALL_RAW_FILES

    lines = []
    lines.append('# NFL Big Data Bowl 2027 — Data Inventory')
    lines.append(f'\n*Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}*\n')

    lines.append('## File Sizes')
    lines.append('| File | Status | Size |')
    lines.append('|------|--------|------|')
    for name, path in ALL_RAW_FILES.items():
        if path.exists():
            size_mb = path.stat().st_size / 1_048_576
            lines.append(f'| {name} | ✓ Found | {size_mb:.1f} MB |')
        else:
            lines.append(f'| {name} | ✗ MISSING | — |')

    def df_summary(name, df):
        if df is None:
            return [f'\n## {name}\n\n**NOT LOADED (file missing)**\n']
        rows = [f'\n## {name}\n']
        rows.append(f'- **Rows**: {len(df):,}')
        rows.append(f'- **Columns**: {df.shape[1]}')
        mem_mb = df.memory_usage(deep=True).sum() / 1_048_576
        rows.append(f'- **Memory**: {mem_mb:.1f} MB')
        if 'nfl_id' in df.columns:
            rows.append(f'- **Unique players**: {df["nfl_id"].nunique():,}')
        if 'game_id' in df.columns:
            rows.append(f'- **Unique games**: {df["game_id"].nunique():,}')
        if 'drill_type' in df.columns:
            rows.append(f'- **Unique drill types**: {df["drill_type"].nunique()}')

        miss = df.isnull().sum()
        miss = miss[miss > 0]
        if not miss.empty:
            rows.append('\n**Missing values:**')
            rows.append('| Column | Missing | Pct |')
            rows.append('|--------|---------|-----|')
            for col, n in miss.items():
                rows.append(f'| {col} | {n:,} | {n/len(df)*100:.1f}% |')
        else:
            rows.append('\n**No missing values.**')

        rows.append('\n**Columns and dtypes:**')
        rows.append('| Column | Dtype |')
        rows.append('|--------|-------|')
        for col, dtype in df.dtypes.items():
            rows.append(f'| {col} | {dtype} |')

        return rows

    for name, df in [
        ('players', players), ('combine_results', combine_results),
        ('combine_tracking', combine_tracking), ('player_play', player_play),
        ('career_successes', career_successes), ('games', games),
    ]:
        lines.extend(df_summary(name, df))

    if join_report:
        lines.append('\n## Join Validation')
        lines.append(f'- Master player table: **{join_report.get("n_players_master", "N/A")}** players')
        lines.append(f'- Players with ALL datasets: **{join_report.get("n_players_with_all_datasets", "N/A")}**')
        for prefix in ['combine_results', 'combine_tracking', 'player_play', 'career_successes']:
            n_key = f'{prefix}_n'
            if n_key in join_report:
                lines.append(f'- {prefix}: **{join_report[n_key]}** players ({join_report.get(f"{prefix}_in_master", "?")} in master)')

    return '\n'.join(lines)


# Build and save report
join_report_data = {}
if players is not None:
    join_report_data = validate_player_joins(
        players=players,
        combine_results=combine_results,
        combine_tracking=combine_tracking,
        player_play=player_play,
        career_successes=career_successes,
    )

report_text = generate_inventory_report(
    players, combine_results, combine_tracking, player_play,
    career_successes, games, join_report_data,
)

report_path = REPORTS_DIR / 'data_inventory.md'
report_path.write_text(report_text, encoding='utf-8')
print(f'Report saved to: {report_path}')
print()
print(report_text[:2000], '...(truncated)')